# 使用 OpenAI Agent SDK 和 Databricks MCP 服务器构建供应链 Copilot

## 解决方案概览

在供应链运营中，智能体可以解答那些直接影响服务水平和收入的问题：我们是否有足够的库存和产能来满足当前需求？哪些环节会出现生产延误，这些延误又会如何向下游传导？哪些工作流调整能将中断降到最低？

![Databricks MCP 界面](../../images/databricks_mcp_ui.png)

本 cookbook 概述了使用 OpenAI Agent SDK 和 Databricks Managed MCP 构建供应链 Copilot 的流程。MCP 让智能体能够查询结构化和非结构化的企业数据，例如库存、销售、供应商数据流、本地事件等，从而获得实时可见性、及早发现物料短缺，并给出前瞻性建议。系统底层由一个编排层支撑，它统一了：
- 对结构化库存、需求和供应商数据的查询
- 针对每个批发商的时间序列预测
- 基于图的原材料需求计算和运输优化
- 支持对非结构化通信进行语义搜索的向量索引邮件归档
- 收入风险计算

完成本指南后，你将部署一个模板，它可以查询分布式数据源和预测模型、突出显示正在出现的瓶颈，并推荐前瞻性措施。它可以回答以下问题：
- 哪些产品依赖 L6HUK 物料？
- 如果我们无法生产出产品 autoclave_1 的预测产量，有多少收入面临风险？
- 目前哪些产品存在延误？
- syringe_1 是否有任何延误？
- syringe_1 需要哪些原材料？
- 以下原材料中是否有任何一种短缺：O4GRQ、Q5U3A、OAIFB 或 58RJD？
- 与批发商 9 相关的延误有哪些？

相关人员可以提交自然语言提示词，并立即获得答案。
本指南将带你逐步在自己的环境中实现这一解决方案。

## 架构

本 cookbook 介绍的架构在你现有的 Databricks 分析工作负载之上叠加了一个 OpenAI 智能体。你可以将 Databricks 组件暴露为可调用的 Unity Catalog 函数。该智能体使用 [OpenAI Agent SDK](https://openai.github.io/openai-agents-python/) 实现，并连接到 [Databricks Managed MCP 服务器](https://docs.databricks.com/aws/en/generative-ai/agent-framework/mcp)。

最终得到的是一个统一的、近实时的对话式界面，可在整个供应链中提供细粒度的预测、动态的库存建议和数据驱动的决策。该架构形成了一个智能体层，充分利用你现有的企业数据（结构化和非结构化）、经典机器学习模型和图分析能力。

![Databricks MCP 架构](../../images/databricks_mcp_architecture.png)

## 设置 Databricks 身份验证

你可以通过向 `~/.databrickscfg` 添加一个配置文件（profile）来设置 Databricks 身份验证。[Databricks 配置文件](https://docs.databricks.com/aws/en/dev-tools/auth/config-profiles)包含 Databricks 进行身份验证所需的设置和其他信息。

代码片段中的 `WorkspaceClient(profile=...)` 调用会读取该配置。它告诉 SDK 要加载哪一组已存储的凭据，因此你的代码永远不需要嵌入令牌。另一种做法是创建 `DATABRICKS_HOST` 和 `DATABRICKS_TOKEN` 等环境变量，但推荐使用 `~/.databrickscfg`。

通过 Settings → Developer → Access tokens → Generate new token 生成一个工作区[个人访问令牌（PAT）](https://docs.databricks.com/aws/en/dev-tools/auth/pat#databricks-personal-access-tokens-for-workspace-users)，然后将其记录到 `~/.databrickscfg` 中。

要创建这个 Databricks 配置文件，可以运行 [Databricks CLI](https://docs.databricks.com/aws/en/dev-tools/cli/) 的 databricks configure 命令，或按以下步骤操作：
- 如果 `~/.databrickscfg` 不存在，请创建它：touch `~/.databrickscfg`
- 打开该文件：`nano ~/.databrickscfg`
- 插入一个配置段，列出工作区 URL 和个人访问令牌（PAT）（之后随时可以添加更多配置）：


```bash
[DEFAULT]
host  = https://dbc-a1b2345c-d6e7.cloud.databricks.com # add your workspace URL here
token = dapi123...    # add your PAT here
```


然后，你可以使用 Databricks CLI 或 SDK 运行 `databricks clusters list` 这个健全性检查命令。如果它在不提示输入凭据的情况下返回了数据，就说明 host 正确且令牌有效。

前提条件是，Databricks 工作区中必须启用 Serverless 计算和 Unity Catalog。

## （可选）设置 Databricks 供应链环境

本 cookbook 可以配合你自己的 Databricks 供应链数据集和分析工作负载使用。

或者，你可以使用 Databricks 供应链优化解决方案加速器（Supply Chain Optimization Solution Accelerator）的定制版本来加快设置。为此，你可以将这个 GitHub [仓库](https://github.com/lara-openai/databricks-supply-chain)克隆到你的 Databricks 工作区，并按照 README [文件](https://github.com/lara-openai/databricks-supply-chain/blob/main/README.md)中的说明操作。运行该解决方案会搭建智能体之后通过 MCP 访问的所有资产，从原始企业数据表和非结构化邮件，到经典机器学习模型和图工作负载。

如果你更希望使用自己的数据集和模型，请确保像加速器中展示的那样，将相关组件封装为 Unity Catalog 函数，并定义一个 Vector Search 索引。你也可以暴露 Genie Spaces。

示例数据模拟了一个真实的制药网络：三家工厂生产 30 种产品，将其运往五个配送中心，每个配送中心为 30-60 个批发商提供服务。该仓库附带了每个产品-批发商组合的时间序列需求数据、配送中心到批发商的映射、工厂到配送中心的成本矩阵、工厂产能上限，以及一个标记了发货延误的邮件归档。

![制药网络](../../images/pharma_network.png)

回答供应链运营问题，需要对上游瓶颈如何在生产、物流和履约环节中层层传导进行建模，以便相关人员缩短交付周期、避免库存积压并控制成本。这些 notebook 将原始数据流转换为受治理、可调用的产物：
- 需求预测与汇总（[notebook 2](https://github.com/lara-openai/databricks-supply-chain/blob/main/02_Fine_Grained_Demand_Forecasting.py)）：使用 Holt-Winters 季节性模型（或任何你偏好的时间序列方法），为每个批发商和配送中心生成未来一周的 SKU 需求。它借助 Pandas UDF 利用 Spark 的并行能力处理大规模预测任务（将你的单节点数据科学代码分发到多个节点上）。然后将预测汇总为每个产品在配送中心层级的总量。输出是一张包含配送中心层级汇总预测的表 product_demand_forecasted。
- 原材料规划（[notebook 3](https://github.com/lara-openai/databricks-supply-chain/blob/main/03_Derive_Raw_Material_Demand.py)）：使用图处理构建产品到物料的映射，沿物料清单（BOM）层级向上传递需求，从而大规模计算组件需求。我们将物料清单转换为图，从而把产品预测转化为精确的原材料需求，生成两张表：raw_material_demand 和 raw_material_supply。
- 运输优化（[notebook 4](https://github.com/lara-openai/databricks-supply-chain/blob/main/04_Optimize_Transportation.py)）：借助 Pandas UDF，在产能和需求约束下最小化工厂到配送中心的运输成本，并将建议输出到 shipment_recommendations。
- 语义邮件搜索（[notebook 6](https://github.com/lara-openai/databricks-supply-chain/blob/main/06_Vector_Search.py)）：使用 OpenAI 嵌入模型将供应链经理的邮件嵌入到向量索引中，支持通过语义查询发现延误和风险信号。

每项洞察都在 [notebook 5](https://github.com/lara-openai/databricks-supply-chain/blob/main/05_Data_Analysis_%26_Functions.py) 和 [notebook 7](https://github.com/lara-openai/databricks-supply-chain/blob/main/07_More_Functions.py) 中被封装为 Unity Catalog（UC）函数，例如 product_from_raw、raw_from_product、revenue_risk、lookup_product_demand、query_unstructured_emails。由于 UC 同样治理表、模型和向量索引，智能体可以在运行时决定是进行预测、追踪 BOM 依赖、评估收入影响、获取历史数据还是搜索邮件，并且始终在调用者的数据访问权限范围内进行。

最终得到的是一个端到端的流水线，它可以预测需求、识别原材料缺口、优化物流、发现隐藏风险，并让分析师提出临时问题、获取延误预警。

在执行完所有 notebook（通过运行 notebook 1）之后，Databricks 环境就准备就绪了，你可以继续构建智能体并将其连接到 Databricks。

## 连接 Databricks MCP 服务器

目前，[MCP 规范](https://openai.github.io/openai-agents-python/mcp/)根据所使用的传输机制定义了三类服务器：
- stdio 服务器作为你的应用的子进程运行。你可以将它们视为在“本地”运行。
- HTTP over SSE 服务器在远程运行。你通过 URL 连接它们。
- Streamable HTTP 服务器在远程运行，使用 MCP 规范中定义的 Streamable HTTP 传输。

[Databricks 托管的 MCP 端点](https://docs.databricks.com/aws/en/generative-ai/agent-framework/mcp)（向量搜索、Unity Catalog 函数、Genie）位于标准 HTTPS URL 之后，并实现了 MCP 规范中定义的 Streamable HTTP 传输。请确保你的工作区已启用 serverless，这样才能连接到 Databricks 托管的 MCP。

## 将 Databricks MCP 服务器集成到 OpenAI 智能体中

OpenAI 智能体的代码可在[这里](https://github.com/openai/openai-cookbook/blob/main/examples/mcp/building-a-supply-chain-copilot-with-agent-sdk-and-databricks-mcp/README.md)获取。首先安装所需的依赖：

In [ ]:
pip install -r requirements.txt

你需要一个 OpenAI API 密钥来安全地访问 API。如果你是 OpenAI API 的新用户，请[注册一个账户](https://platform.openai.com/signup)。你可以按照[这些步骤](https://platform.openai.com/docs/libraries?project_id=proj_2NqyDkmG63zyr3TzOh64F2ac#create-and-export-an-api-key)创建密钥，并将其存放在安全的位置。

本 cookbook 展示了如何使用 FastAPI 提供该智能体服务，并通过 React 界面进行聊天。不过，`main.py` 被设置为一个自包含的 REPL，因此在安装所需依赖并设置好必要的凭据（包括如上所述的 Databricks host 和个人访问令牌）之后，你只需一条命令就可以直接从命令行运行智能体：

In [ ]:
python main.py

[main.py](https://github.com/openai/openai-cookbook/blob/main/examples/mcp/building-a-supply-chain-copilot-with-agent-sdk-and-databricks-mcp/main.py) 文件负责编排智能体逻辑，它使用 OpenAI Agent SDK，并将 Databricks MCP 向量搜索端点和 Unity Catalog 函数暴露为可调用的工具。它首先读取指向目标 catalog、schema 和 Unity Catalog（UC）函数路径的环境变量，然后暴露两个工具：vector_search（查询 Databricks Vector Search 索引）和 uc_function（通过 MCP 执行 Unity Catalog 函数）。两个工具都通过 httpx 发出经过身份验证的 POST 请求，并返回来自 Databricks REST API 的原始 JSON。两个辅助函数都通过 _databricks_ctx() 工具函数（由 DatabricksOAuthClientProvider 支持）获取工作区 host 和个人访问令牌，并使用 httpx 发出经过身份验证的 POST 请求，返回原始 JSON 响应。

在 run_agent() 中，脚本实例化了一个名为“Assistant”的智能体，其范围被严格限定在供应链话题内。每个回复都必须调用两个已注册工具之一，护栏会强制智能体拒绝物流、库存、采购或预测以外的任何请求。每个用户提示词都在一个 SDK trace 上下文中处理。一个简单的 REPL 驱动整个交互：用户输入被包装在一个 OpenTelemetry 风格的 trace 中，通过 Runner.run 分发，然后打印最终答案（或护栏触发的致歉）。程序通过 main() 中的 asyncio.run 调用启动，使整个流程完全异步且非阻塞。

In [0]:
"""
CLI assistant that uses Databricks MCP Vector Search and UC Functions via the OpenAI Agents SDK.
"""

import asyncio
import os
import httpx
from typing import Dict, Any
from agents import Agent, Runner, function_tool, gen_trace_id, trace
from agents.exceptions import (
    InputGuardrailTripwireTriggered,
    OutputGuardrailTripwireTriggered,
)
from agents.model_settings import ModelSettings
from databricks_mcp import DatabricksOAuthClientProvider
from databricks.sdk import WorkspaceClient
from supply_chain_guardrails import supply_chain_guardrail

CATALOG = os.getenv("MCP_VECTOR_CATALOG", "main") # override catalog, schema, functions_path name if your data assets sit in a different location
SCHEMA = os.getenv("MCP_VECTOR_SCHEMA", "supply_chain_db")
FUNCTIONS_PATH = os.getenv("MCP_FUNCTIONS_PATH", "main/supply_chain_db")
DATABRICKS_PROFILE = os.getenv("DATABRICKS_PROFILE", "DEFAULT") # override if using a different profile name 
HTTP_TIMEOUT = 30.0  # seconds


async def _databricks_ctx():
    """Return (workspace, PAT token, base_url)."""
    ws = WorkspaceClient(profile=DATABRICKS_PROFILE)
    token = DatabricksOAuthClientProvider(ws).get_token()
    return ws, token, ws.config.host


@function_tool
async def vector_search(query: str) -> Dict[str, Any]:
    """Query Databricks MCP Vector Search index."""
    ws, token, base_url = await _databricks_ctx()
    url = f"{base_url}/api/2.0/mcp/vector-search/{CATALOG}/{SCHEMA}"
    headers = {"Authorization": f"Bearer {token}"}
    async with httpx.AsyncClient(timeout=HTTP_TIMEOUT) as client:
        resp = await client.post(url, json={"query": query}, headers=headers)
        resp.raise_for_status()
        return resp.json()


@function_tool
async def uc_function(function_name: str, params: Dict[str, Any]) -> Dict[str, Any]:
    """Invoke a Databricks Unity Catalog function with parameters."""
    ws, token, base_url = await _databricks_ctx()
    url = f"{base_url}/api/2.0/mcp/functions/{FUNCTIONS_PATH}"
    headers = {"Authorization": f"Bearer {token}"}
    payload = {"function": function_name, "params": params}
    async with httpx.AsyncClient(timeout=HTTP_TIMEOUT) as client:
        resp = await client.post(url, json=payload, headers=headers)
        resp.raise_for_status()
        return resp.json()


async def run_agent():
    agent = Agent(
        name="Assistant",
        instructions="You are a supply-chain assistant for Databricks MCP; you must answer **only** questions that are **strictly** about supply-chain data, logistics, inventory, procurement, demand forecasting, etc; for every answer you must call one of the registered tools; if the user asks anything not related to supply chain, reply **exactly** with 'Sorry, I can only help with supply-chain questions'.",
        tools=[vector_search, uc_function],
        model_settings=ModelSettings(model="gpt-4o", tool_choice="required"),
        output_guardrails=[supply_chain_guardrail],
    )

    print("Databricks MCP assistant ready. Type a question or 'exit' to quit.")

    while True:
        user_input = input("You: ").strip()
        if user_input.lower() in {"exit", "quit"}:
            break

        trace_id = gen_trace_id()
        with trace(workflow_name="Databricks MCP Agent", trace_id=trace_id):
            try:
                result = await Runner.run(starting_agent=agent, input=user_input)
                print("Assistant:", result.final_output)
            except InputGuardrailTripwireTriggered:
                print("Assistant: Sorry, I can only help with supply-chain questions.")
            except OutputGuardrailTripwireTriggered:
                print("Assistant: Sorry, I can only help with supply-chain questions.")


def main():
    asyncio.run(run_agent())


if __name__ == "__main__":
    main()

[databricks_mcp.py](https://github.com/openai/openai-cookbook/blob/main/examples/mcp/building-a-supply-chain-copilot-with-agent-sdk-and-databricks-mcp/databricks_mcp.py) 是一个专注的身份验证抽象：它从给定的 WorkspaceClient（ws.config.token）中获取我们之前创建的个人访问令牌，并让应用的其余部分无需关心 Databricks 特有的 OAuth 逻辑。通过将所有令牌处理细节限制在这一个模块中，未来 Databricks 身份验证方案的任何变化都只需更新这个文件即可适配。


In [ ]:
"""
Databricks OAuth client provider for MCP servers.
"""

class DatabricksOAuthClientProvider:
    def __init__(self, ws):
        self.ws = ws

    def get_token(self):
        # For Databricks SDK >=0.57.0, token is available as ws.config.token
        return self.ws.config.token

[supply_chain_guardrails.py](https://github.com/openai/openai-cookbook/blob/main/examples/mcp/building-a-supply-chain-copilot-with-agent-sdk-and-databricks-mcp/supply_chain_guardrails.py) 实现了一个轻量级的输出护栏：它启动第二个智能体（“Supply‑chain check”）来对候选答案进行分类。主智能体将草拟的回复交给这个检查器，检查器返回一个带有布尔字段 is_supply_chain 的 Pydantic 对象。如果该标志为 false，护栏就会触发 tripwire，调用方随即换成一条拒绝回复。

In [ ]:
"""
Output guardrail that blocks answers not related to supply-chain topics.
"""
from __future__ import annotations

from pydantic import BaseModel
from agents import Agent, Runner, GuardrailFunctionOutput
from agents import output_guardrail
from agents.run_context import RunContextWrapper

class SupplyChainCheckOutput(BaseModel):
    reasoning: str
    is_supply_chain: bool


guardrail_agent = Agent(
    name="Supply-chain check",
    instructions=(
        "Check if the text is within the domain of supply-chain analytics and operations "
        "Return JSON strictly matching the SupplyChainCheckOutput schema"
    ),
    output_type=SupplyChainCheckOutput,
)


@output_guardrail
async def supply_chain_guardrail(
    ctx: RunContextWrapper, agent: Agent, output
) -> GuardrailFunctionOutput:
    """Output guardrail that blocks non-supply-chain answers"""
    text = output if isinstance(output, str) else getattr(output, "response", str(output))
    result = await Runner.run(guardrail_agent, text, context=ctx.context)
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=not result.final_output.is_supply_chain,
    )

## 使用 FastAPI 提供智能体服务

要启动后端（FastAPI），请运行以下命令：

In [ ]:
python -m uvicorn api_server:app --reload --port 8000

API 将在 http://localhost:8000 上提供（FastAPI 文档请访问： http://localhost:8000/docs ）。

[api_server.py](https://github.com/openai/openai-cookbook/blob/main/examples/mcp/building-a-supply-chain-copilot-with-agent-sdk-and-databricks-mcp/api_server.py) 是一个 FastAPI 后端，它将你的智能体暴露为一个流式的 /chat API 端点。启动时，它会配置 CORS，以便本地前端可以与之通信，然后定义 `build_mcp_servers()`：该函数向调用者的 Databricks 工作区进行身份验证，构建两个 HTTP“服务器工具”（一个用于向量搜索，一个用于 Unity Catalog 函数），并预先建立连接以实现低延迟使用。每个发往 /chat 的 POST 请求都包含一条用户消息。处理程序会启动一个全新的智能体，其 mcp_servers 列表由这些流式工具填充，并且模型在每一轮都被强制调用工具。

In [0]:
"""
FastAPI wrapper that exposes the agent as a streaming `/chat` endpoint.
"""

import os
import asyncio
import logging
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from agents.exceptions import (
    InputGuardrailTripwireTriggered,
    OutputGuardrailTripwireTriggered,
)
from agents import Agent, Runner, gen_trace_id, trace
from agents.mcp import MCPServerStreamableHttp, MCPServerStreamableHttpParams
from agents.model_settings import ModelSettings
from databricks_mcp import DatabricksOAuthClientProvider
from databricks.sdk import WorkspaceClient

from supply_chain_guardrails import supply_chain_guardrail

CATALOG = os.getenv("MCP_VECTOR_CATALOG", "main")
SCHEMA = os.getenv("MCP_VECTOR_SCHEMA", "supply_chain_db")
FUNCTIONS_PATH = os.getenv("MCP_FUNCTIONS_PATH", "main/supply_chain_db")
DATABRICKS_PROFILE = os.getenv("DATABRICKS_PROFILE", "DEFAULT")
HTTP_TIMEOUT = 30.0  # seconds

app = FastAPI()

# Allow local dev front‑end
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:5173"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class ChatRequest(BaseModel):
    message: str


async def build_mcp_servers():
    """Initialise Databricks MCP vector & UC‑function servers."""
    ws = WorkspaceClient(profile=DATABRICKS_PROFILE)
    token = DatabricksOAuthClientProvider(ws).get_token()

    base = ws.config.host
    vector_url = f"{base}/api/2.0/mcp/vector-search/{CATALOG}/{SCHEMA}"
    fn_url = f"{base}/api/2.0/mcp/functions/{FUNCTIONS_PATH}"

    async def _proxy_tool(request_json: dict, url: str):
        import httpx

        headers = {"Authorization": f"Bearer {token}"}
        async with httpx.AsyncClient(timeout=HTTP_TIMEOUT) as client:
            resp = await client.post(url, json=request_json, headers=headers)
            resp.raise_for_status()
            return resp.json()

    headers = {"Authorization": f"Bearer {token}"}

    servers = [
        MCPServerStreamableHttp(
            MCPServerStreamableHttpParams(
                url=vector_url,
                headers=headers,
                timeout=HTTP_TIMEOUT,
            ),
            name="vector_search",
            client_session_timeout_seconds=60,
        ),
        MCPServerStreamableHttp(
            MCPServerStreamableHttpParams(
                url=fn_url,
                headers=headers,
                timeout=HTTP_TIMEOUT,
            ),
            name="uc_functions",
            client_session_timeout_seconds=60,
        ),
    ]

    # Ensure servers are initialized before use
    await asyncio.gather(*(s.connect() for s in servers))
    return servers


@app.post("/chat")
async def chat_endpoint(req: ChatRequest):
    try:
        servers = await build_mcp_servers()

        agent = Agent(
            name="Assistant",
            instructions="Use the tools to answer the questions.",
            mcp_servers=servers,
            model_settings=ModelSettings(tool_choice="required"),
            output_guardrails=[supply_chain_guardrail],
        )

        trace_id = gen_trace_id()

        async def agent_stream():
            logging.info(f"[AGENT_STREAM] Input message: {req.message}")
            try:
                with trace(workflow_name="Databricks MCP Example", trace_id=trace_id):
                    result = await Runner.run(starting_agent=agent, input=req.message)
                    logging.info(f"[AGENT_STREAM] Raw agent result: {result}")
                    try:
                        logging.info(
                            f"[AGENT_STREAM] RunResult __dict__: {getattr(result, '__dict__', str(result))}"
                        )
                        raw_responses = getattr(result, "raw_responses", None)
                        logging.info(f"[AGENT_STREAM] RunResult raw_responses: {raw_responses}")
                    except Exception as log_exc:
                        logging.warning(f"[AGENT_STREAM] Could not log RunResult details: {log_exc}")
                    yield result.final_output
            except InputGuardrailTripwireTriggered:
                # Off-topic question denied by guardrail
                yield "Sorry, I can only help with supply-chain questions."
            except OutputGuardrailTripwireTriggered:
                # Out-of-scope answer blocked by guardrail
                yield "Sorry, I can only help with supply-chain questions."
            except Exception:
                logging.exception("[AGENT_STREAM] Exception during agent run")
                yield "[ERROR] Exception during agent run. Check backend logs for details."

        return StreamingResponse(agent_stream(), media_type="text/plain")

    except Exception:
        logging.exception("chat_endpoint failed")
        return StreamingResponse(
            (line.encode() for line in ["Internal server error 🙈"]),
            media_type="text/plain",
            status_code=500,
        )

在智能体推理和调用 MCP 工具的同时，该端点会将 token 以流式方式返回给浏览器。

## 通过 React 聊天界面与用户交互

在另一个终端中，运行以下命令启动前端（React 界面）：

In [ ]:
cd ui
npm install
npm run dev

应用将在 http://localhost:5173 上提供

[/ui 文件夹](https://github.com/openai/openai-cookbook/blob/main/examples/mcp/building-a-supply-chain-copilot-with-agent-sdk-and-databricks-mcp/ui)中的 React 聊天界面提供了一个用户友好的 Web 界面，用于与后端智能体交互。它包含用于显示对话历史的组件，以及用于发送消息的文本输入框。

当用户提交消息时，界面会将其发送到后端的 /chat 端点，并实时流式接收智能体的回复，随着新内容的到达不断更新聊天窗口。该设计强调对话式体验，让用户可以在一个响应迅速、交互性强的 Web 应用中，轻松地向由 Databricks 驱动的智能体提问并获得答案。

具体来说，[ChatUI.jsx](https://github.com/openai/openai-cookbook/blob/main/examples/mcp/building-a-supply-chain-copilot-with-agent-sdk-and-databricks-mcp/ui/src/components/ChatUI.jsx) 文件包含聊天界面的核心逻辑，包括如何将用户消息发送到后端，以及如何实时处理和显示来自智能体的流式回复。

In [ ]:
"""
Code snippet handling the token stream coming from the FastAPI /chat endpoint.
"""
const reader = response.body.getReader();
while (true) {
  const { done, value } = await reader.read();
  if (done) break;
  assistantMsg.text += new TextDecoder().decode(value);
  setMessages(m => {
    const copy = [...m];
    copy[copy.length - 1] = { ...assistantMsg };
    return copy;
  });
}

界面会在智能体的回复到达时以流式方式显示，带来流畅的实时聊天体验。重点展示这一点，可以让读者清楚地看到界面是如何从后端智能体获得交互式、对话式反馈的。

![Databricks MCP 界面](../../images/databricks_mcp_ui.png)

## 向应用提问

访问 http://localhost:5173 并尝试以下提示词：
- What products are dependent on L6HUK material?
- How much revenue is at risk if we can’t produce the forecasted amount of product autoclave_1?
- Which products have delays right now?
- Are there any delays with syringe_1? 
- What raw materials are required for syringe_1?
- Are there any shortages with one of the following raw materials: O4GRQ, Q5U3A, OAIFB or 58RJD?
- What are the delays associated with wholesaler 9?
智能体会调用相关工具，并为用户整理出一个有依据的答案。


## 在 OpenAI API 仪表盘中追踪智能体调用

在 OpenAI API [仪表盘](https://platform.openai.com/prompts)中，你可以打开 Traces 视图，查看智能体调用的每一个函数。在下面的示例中，智能体先调用 raw_from_product 获取与特定产品关联的物料，然后调用 revenue_risk 估算短缺对收入的影响。

![追踪仪表盘](../../images/tracing_dashboard_databricks_mcp.png)

## 后续步骤

* 你可以考虑添加多轮对话能力
* 如果你想将这套设置适配到自己的工作区，也可以添加 Genie Space MCP 服务器

## 参考资料
- Databricks Managed MCP [文档](https://docs.databricks.com/aws/en/generative-ai/agent-framework/mcp)
- OpenAI Agent SDK [文档](https://openai.github.io/openai-agents-python/)
- OpenAI Agent 护栏[文档](https://openai.github.io/openai-agents-python/guardrails/)
- `openai-agents-python` 示例[代码片段](https://github.com/openai/openai-agents-python/tree/main/examples/mcp/streamablehttp_example)